# Pipeline a Gran Escala en Google Colab: Detección de Radicalización en la Manosfera
### Proyecto CC219 - Aplicaciones de Data Science
Este notebook permite descargar y procesar millones de publicaciones del dataset de Zenodo (DOI: 10.5281/zenodo.4007913) aprovechando el ancho de banda y la GPU de Google Colab sin sobrecargar el repositorio local de GitHub.

In [ ]:
# 1. Verificacion de entorno y GPU
!nvidia-smi
!pip install -q transformers datasets accelerate sentencepiece scikit-learn
import os, json, zipfile, torch
import pandas as pd
import numpy as np
print("PyTorch version:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

## 2. Descarga Directa desde Zenodo a Alta Velocidad
En la infraestructura de Google Colab la descarga de ~5.8 GB toma habitualmente menos de 30-40 segundos.

In [ ]:
# Descargar ndjson.zip (22.1 millones de posts de Reddit, incluyendo r/Braincels)
!curl -L "https://zenodo.org/api/records/4007913/files/ndjson.zip/content" -o ndjson.zip

# Descomprimir solo los archivos necesarios
!unzip -q ndjson.zip -d ./data_large
!ls -lh ./data_large

## 3. Ingestión y Filtrado en Streaming (Chunking)
Procesamos el archivo `reddit.ndjson` línea por línea para filtrar exclusivamente publicaciones de subreddits clave como `Braincels` o `Incels`, y retener usuarios con trayectorias longitudinales.

In [ ]:
ndjson_path = './data_large/ndjson/reddit.ndjson' # ajustar segun estructura descomprimida
target_subreddits = {'Braincels', 'Incels', 'TheRedPill', 'MGTOW'}

records = []
max_posts = 100000  # Escalar a 100,000 registros para entrenamiento
count = 0

print("Extrayendo publicaciones relevantes...")
with open(ndjson_path, 'r', encoding='utf-8', errors='ignore') as f:
    for line in f:
        try:
            post = json.loads(line)
            sub = post.get('subreddit', '')
            if sub in target_subreddits:
                text = post.get('text_post', '').strip()
                if len(text.split()) >= 5: # filtrar mensajes con mas de 5 palabras
                    records.append({
                        'author': post.get('author'),
                        'date_post': post.get('date_post'),
                        'subreddit': sub,
                        'text_post': text,
                        'id_post': post.get('id_post')
                    })
                    count += 1
                    if count >= max_posts:
                        break
        except Exception:
            continue

df_large = pd.DataFrame(records)
print(f"Extraccion completada: {len(df_large)} publicaciones de {df_large['author'].nunique()} autores.")
df_large.head()

## 4. Filtrado Longitudinal para Modelar la Espiral de Radicalización ("Spiraling")
Seleccionamos autores que tengan al menos 15 publicaciones ordenadas cronológicamente para estudiar su transición hacia el extremismo.

In [ ]:
# Ordenar temporalmente por autor
df_large['date_post'] = pd.to_numeric(df_large['date_post'], errors='coerce')
df_large = df_large.dropna(subset=['date_post']).sort_values(by=['author', 'date_post'])

# Conteo de posts por autor
author_counts = df_large['author'].value_counts()
prolific_users = author_counts[author_counts >= 15].index
df_spiraling = df_large[df_large['author'].isin(prolific_users)].copy()

# Asignar orden de secuencia de cada usuario
df_spiraling['user_post_seq'] = df_spiraling.groupby('author').cumcount() + 1
print(f"Dataset longitudinal: {len(df_spiraling)} publicaciones correspondientes a {len(prolific_users)} usuarios activos.")

## 5. Fine-Tuning con Transformer Preentrenado (HateBERT / RoBERTa)
Utilizamos la GPU de Google Colab para entrenar un modelo basado en `GroNLP/hate-bert` o `cardiffnlp/twitter-roberta-base-hate` para clasificar la severidad de riesgo y odio.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from datasets import Dataset

MODEL_NAME = "GroNLP/hate-bert"
print(f"Cargando tokenizer para {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Ejemplo de tokenizacion por lotes
sample_texts = df_spiraling['text_post'].iloc[:1000].tolist()
tokens = tokenizer(sample_texts, padding=True, truncation=True, max_length=128, return_tensors='pt')
print("Lote tokenizado exitosamente en memoria con forma:", tokens['input_ids'].shape)

## 6. Guardar Dataset Muestral y Métricas Finales para el Reporte
Al terminar la ejecución en Colab, podemos exportar únicamente los pesos del modelo final y las tablas de resultados a Google Drive para agregarlas al documento final del curso.

In [ ]:
# Exportar resumen a Google Drive o descarga directa
df_spiraling.to_parquet('incels_longitudinal_100k.parquet', compression='snappy')
print("Archivo exportado: incels_longitudinal_100k.parquet listo para analisis posterior.")